# 教程 3：按请求路由的 LoRA

**场景**：邮箱类请求走一路 LoRA，普通请求走另一路；路由规则是你自己的正则。
在此之上再看两件事：用判别器 MLP 对两路 LoRA 做加权混合；prefill 期间按
wavefront / Jacobi 两种调度逐步学习。

教程 2 的训练头**不改模型**（只引导采样）。LoRA 不同 —— 它直接改层的权重。
这在 PLLM 里是另一种效果：**overlay**。本篇分四步：

1. 最小 overlay：一个低秩补丁挂在某层 MLP 上；
2. 谁和谁共享一份 LoRA：`keys` 与 route 键；
3. 判别器加权混合：只是"状态里多几个字段"；
4. prefill 内的逐步学习：wavefront 与 Jacobi。

## 第一步：最小 overlay

overlay 任务在 `Phase.MODEL_UPDATE` 时机运行，它的输入不是 hidden，而是
**被捕获的算子三元组**：目标层的输入 `x`、输出 `y`、原始权重 `base`。
下面的 update 做两件小事：算出加了低秩补丁后的输出（模型立刻用它继续前向），
再做一步 Hebbian 风格的小更新。

两个签名约束值得记住：

- overlay 状态字段用 `pllm.Buffer` 声明，形状可以依赖目标参数的尺寸
  （`inputs`/`outputs` 是目标权重的入/出维度）；
- buffer 状态**必须零初始化** —— 初始行为因此严格等于无补丁。

In [1]:
import os
os.environ.setdefault("VLLM_ENABLE_V1_MULTIPROCESSING", "0")

import gc
import re
from typing import Annotated

import torch
import pllm
from pllm import TensorUpdate
from vllm import SamplingParams
from pllm.config import VllmDType, ModelConfig
from pllm.contracts.parameter_overlay import ParameterModuleKind, ParameterRole

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

def config(max_num_seqs=2):
    return ModelConfig(MODEL, VllmDType.BFLOAT16, 96, max_num_seqs, 96, 0.2, seed=7)

def cleanup():
    gc.collect()
    torch.cuda.empty_cache()

def text_of(handle, prompt, max_tokens=24, **kwargs):
    return handle.generate([prompt], SamplingParams(max_tokens=max_tokens,
        temperature=0.0, ignore_eos=True), use_tqdm=False, **kwargs)[0].outputs[0].text

LR = 0.1
RANK = 2
TARGET9 = tuple(pllm.ParameterTarget(layer, ParameterModuleKind.MLP,
    ParameterRole.DOWN_PROJECTION_WEIGHT) for layer in (7, 8, 9))


class MyFirstLoRA(pllm.Workload):
    """7-9 层 MLP down-projection 上的 rank-2 低秩补丁，decode 时每步更新。"""

    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, dtype=torch.bfloat16)

    @pllm.on(pllm.Phase.MODEL_UPDATE)
    @pllm.overlay(state="factors", targets=TARGET9,
                  operator=pllm.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, decode=True)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> TensorUpdate:
        output = x @ (base + b @ a).T                  # 模型实际使用的前向
        a.add_((LR * y[:, :RANK].T @ x).to(a))         # Hebbian 风格小更新
        b.add_((LR * (y.T @ y)[:, :RANK]).to(b))
        return TensorUpdate(output, changed=True, update_count=1, trained_rows=x.shape[0])


lora = pllm.make_llm(workload=MyFirstLoRA(), config=config(max_num_seqs=1))
print(text_of(lora, "The sky is"))
print("updates:", lora.worker_state().diagnostics.update_count)

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 12:39:40,874	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 12:39:41 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:39:42 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:39:43 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:39:43 [model.py:1883] Using max model len 96


... (26 outputs left out)

INFO 09-10 12:39:50 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


INFO 09-10 12:39:51 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.61 s


INFO 09-10 12:39:54 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


 a blue rolling '..', '..', '..',יהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהןיהן
updates: 72


## 第二步：谁和谁共享 —— `keys` 与 route

默认情况下请求状态按请求隔离（keys 是请求身份）。把 `keys` 改成
`(RequestKeyField.ROUTE,)` 之后，**路由键相同的请求共享一条 lineage**：
第一个请求退役后 lineage 休眠保留，下一个同键请求直接加入它，读到累积的训练。

路由键完全由你计算 —— 在 `generate(..., routes=(...))` 里传入。
这里用一个邮箱正则：带邮箱的 prompt 归 "email"，其余归 "plain"。

三个可检验的断言：

- 同一 route 第二次生成 ≠ 第一次（读到了累积训练）；
- 另一个 route 第二次生成也 ≠ 它的第一次（它自己也积累了）；
- 全新 route 的第一次生成 == 任何训练发生之前的基线（隔离严格）。

In [2]:
class RoutedLoRA(MyFirstLoRA):
    # 唯一的区别：lineage 按 route 键组织，跨请求共享
    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, dtype=torch.bfloat16, keys=(pllm.RequestKeyField.ROUTE,))


EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w]+")

def route_of(prompt: str) -> str:
    return "email" if EMAIL.search(prompt) else "plain"


lora.close()
cleanup()
lora = pllm.make_llm(workload=RoutedLoRA(), config=config(max_num_seqs=4))
email_prompt = "Write a reply to alice@example.com about the meeting"
plain_prompt = "The sky is"

plain_1 = text_of(lora, plain_prompt, routes=("plain",))
email_1 = text_of(lora, email_prompt, routes=(route_of(email_prompt),))
plain_2 = text_of(lora, plain_prompt, routes=("plain",))
email_2 = text_of(lora, email_prompt, routes=(route_of(email_prompt),))
fresh = text_of(lora, plain_prompt, routes=("another",))

print("email 第 1 次:", email_1)
print("email 第 2 次:", email_2)
assert email_1 != email_2        # 同 route：读到累积的训练
assert plain_1 != plain_2        # plain 自己也积累了
assert fresh == plain_1          # 新 route 从零开始，不受他人训练影响
print("route 共享与隔离都得证")

INFO 09-10 12:39:55 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:39:55 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:39:56 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:39:56 [model.py:1883] Using max model len 96


INFO 09-10 12:39:56 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:39:58 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=96, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (12 outputs left out)

WARNING 09-10 12:40:00 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _prepare_pos_seq_lens_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


WARNING 09-10 12:40:00 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _gather_block_tables_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


INFO 09-10 12:40:00 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.91 s


email 第 1 次: .. '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..',
email 第 2 次:  '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..', '..',
route 共享与隔离都得证


## 第三步：判别器加权混合 —— 只是"多几个字段"

两路 LoRA 不该非此即彼时，让一个小判别器 MLP 按当前 hidden 给两路打分，
softmax 加权混合。**这不需要任何新机制**：一个状态里放两路因子加判别器，
`forward` 里写混合数学，训练照旧（教程 2 的骨架）。

注意参数依然**按行就位**：`a1` 读出来是 `(行数, H, RANK)`，所以用 `bmm`。

In [3]:
class MixtureLoRA(pllm.Workload):
    """判别器对两路 LoRA 头的输出做 softmax 加权；所有字段一起训练。"""

    mixture = pllm.RequestState(
        {"a1": lambda m: (m.hidden_size, RANK), "b1": lambda m: (RANK, m.hidden_size),
         "a2": lambda m: (m.hidden_size, RANK), "b2": lambda m: (RANK, m.hidden_size),
         "disc_w": lambda m: (m.hidden_size, 2), "disc_b": (2,)},
        dtype=torch.float32,
        initialize=lambda tensors, generator: [
            torch.nn.init.normal_(tensor, std=0.02, generator=generator)
            for tensor in tensors.values()],
        optimizer=pllm.AdamW(lr=3e-3),
        keys=(pllm.RequestKeyField.ROUTE,))

    def forward(self, h, a1, b1, a2, b2, disc_w, disc_b):
        x = h.float().unsqueeze(1)                                            # (rows, 1, H)
        weights = torch.softmax(torch.bmm(x, disc_w).squeeze(1) + disc_b, -1) # (rows, 2)
        out1 = torch.bmm(torch.bmm(x, a1), b1).squeeze(1)
        out2 = torch.bmm(torch.bmm(x, a2), b2).squeeze(1)
        return weights[:, 0:1] * out1 + weights[:, 1:2] * out2

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("mixture")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, h: Annotated[torch.Tensor, pllm.Hidden(-1)],
                a1: Annotated[torch.Tensor, pllm.Parameter("mixture", "a1")],
                b1: Annotated[torch.Tensor, pllm.Parameter("mixture", "b1")],
                a2: Annotated[torch.Tensor, pllm.Parameter("mixture", "a2")],
                b2: Annotated[torch.Tensor, pllm.Parameter("mixture", "b2")],
                disc_w: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_w")],
                disc_b: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_b")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("mixture")]) -> torch.Tensor:
        return self.forward(h, a1, b1, a2, b2, disc_w, disc_b)

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="mixture", reduction="state_mean")
    def learn(self, h: Annotated[torch.Tensor, pllm.Hidden(-1)],
              target: Annotated[torch.Tensor, pllm.Hidden(0)],
              a1: Annotated[torch.Tensor, pllm.Parameter("mixture", "a1")],
              b1: Annotated[torch.Tensor, pllm.Parameter("mixture", "b1")],
              a2: Annotated[torch.Tensor, pllm.Parameter("mixture", "a2")],
              b2: Annotated[torch.Tensor, pllm.Parameter("mixture", "b2")],
              disc_w: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_w")],
              disc_b: Annotated[torch.Tensor, pllm.Parameter("mixture", "disc_b")],
              groups: Annotated[torch.Tensor, pllm.StateIndex("mixture")]) -> torch.Tensor:
        losses = (self.forward(h, a1, b1, a2, b2, disc_w, disc_b) - target.float()).square().mean(1)
        counts = torch.bincount(groups).to(losses)
        return losses.new_zeros(counts.shape).index_add(0, groups, losses) / counts


lora.close()
cleanup()
mixture = pllm.make_llm(workload=MixtureLoRA(), config=config(max_num_seqs=4))
print("email:", text_of(mixture, email_prompt, routes=("email",)))
print("plain:", text_of(mixture, plain_prompt, routes=("plain",)))
d = mixture.worker_state().diagnostics
print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
mixture.close()

INFO 09-10 12:40:01 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:40:02 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:40:03 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:40:03 [model.py:1883] Using max model len 96


INFO 09-10 12:40:03 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:40:04 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=96, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (8 outputs left out)

INFO 09-10 12:40:06 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.92 s


WARNING 09-10 12:40:06 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


email: . Alice is a software engineer who works at a tech company. She is currently working on a project that involves developing a


plain:  a beautiful place. It is a place of endless possibilities. The sky is a place of endless possibilities for people to explore
losses: 46 updates: 46 fallback: 0


## 第四步：prefill 内的逐步学习 —— wavefront 与 Jacobi

prefill 一次处理一批 token。按 chunk 切开后有两种经典调度：

- **wavefront**：chunk k+1 的前向立即看到 chunk k 的更新（边算边改）；
- **Jacobi**：所有 chunk 都对同一输入版本求值，更新在最后一次性施加。

两者只是 `update` 里的循环写法不同。为了让它非平凡，更新规则里
`a` 的增量与当前因子无关（chunk 1 就移动），`b` 的增量依赖当前 `a`
（wavefront 逐步复合，Jacobi 停在输入版本）。

In [4]:
LAYERS = (1, 3, 5, 7, 9)
TARGETS = tuple(pllm.ParameterTarget(layer, ParameterModuleKind.MLP,
    ParameterRole.DOWN_PROJECTION_WEIGHT) for layer in LAYERS)
CHUNK = 8


class Wavefront(pllm.Workload):
    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, dtype=torch.bfloat16)

    @pllm.on(pllm.Phase.MODEL_UPDATE)
    @pllm.overlay(state="factors", targets=TARGETS,
                  operator=pllm.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, prefill=True, decode=False)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> TensorUpdate:
        outputs = []
        for start in range(0, x.shape[0], CHUNK):
            xc, yc = x[start:start + CHUNK], y[start:start + CHUNK]
            outputs.append(xc @ (base + b @ a).T)         # 当前因子立即生效
            a.add_((LR * yc[:, :RANK].T @ xc).to(a))
            b.add_((LR * yc.T @ (xc @ a.T)).to(b))        # 看到刚更新的 a
        return TensorUpdate(torch.cat(outputs), changed=True,
                            update_count=len(outputs), trained_rows=x.shape[0])


class Jacobi(pllm.Workload):
    factors = pllm.RequestState({
        "a": pllm.Buffer(lambda inputs, outputs: (RANK, inputs)),
        "b": pllm.Buffer(lambda inputs, outputs: (outputs, RANK)),
    }, dtype=torch.bfloat16)

    @pllm.on(pllm.Phase.MODEL_UPDATE)
    @pllm.overlay(state="factors", targets=TARGETS,
                  operator=pllm.ParameterOverlayOperator.LOW_RANK_ADD,
                  factors=("a", "b"), scale=1.0, prefill=True, decode=False)
    def update(self, x: Annotated[torch.Tensor, pllm.Operand.INPUT],
               y: Annotated[torch.Tensor, pllm.Operand.TARGET],
               base: Annotated[torch.Tensor, pllm.Operand.WEIGHT],
               a: Annotated[torch.Tensor, pllm.Parameter("factors", "a")],
               b: Annotated[torch.Tensor, pllm.Parameter("factors", "b")]) -> TensorUpdate:
        outputs = [x[start:start + CHUNK] @ (base + b @ a).T
                   for start in range(0, x.shape[0], CHUNK)]  # 全部对输入版本求值
        grad_a = torch.zeros_like(a)
        grad_b = torch.zeros_like(b)
        for start in range(0, x.shape[0], CHUNK):
            xc, yc = x[start:start + CHUNK], y[start:start + CHUNK]
            grad_a += yc[:, :RANK].T @ xc
            grad_b += yc.T @ (xc @ a.T)                   # a 还是输入版本
        a.add_((LR * grad_a).to(a))
        b.add_((LR * grad_b).to(b))
        return TensorUpdate(torch.cat(outputs), changed=True,
                            update_count=len(outputs), trained_rows=x.shape[0])


cleanup()
long_prompt = "The quick brown fox jumps over the lazy dog near the river bank"
wave = pllm.make_llm(workload=Wavefront(), config=config(max_num_seqs=1))
text_wave = text_of(wave, long_prompt)
updates_wave = wave.worker_state().diagnostics.update_count
wave.close()
cleanup()
jac = pllm.make_llm(workload=Jacobi(), config=config(max_num_seqs=1))
text_jac = text_of(jac, long_prompt)
updates_jac = jac.worker_state().diagnostics.update_count
jac.close()

print("wavefront:", text_wave, "updates:", updates_wave)
print("jacobi:   ", text_jac, "updates:", updates_jac)
assert text_wave != text_jac and updates_wave == updates_jac > 0

INFO 09-10 12:40:08 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 96, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 96, 'max_num_seqs': 1, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 12:40:08 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 12:40:09 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 12:40:09 [model.py:1883] Using max model len 96


INFO 09-10 12:40:09 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 12:40:10 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=96, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (24 outputs left out)

INFO 09-10 12:40:18 [model_runner.py:329] Model loading took 0.93 GiB and 1.060253 seconds


INFO 09-10 12:40:18 [gpu_worker.py:789] Free memory on device (11.6/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.96 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3852515328` (3.59 GiB) to fit into requested memory, or `--kv-cache-memory=11269093376` (10.5 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.73 GiB.


INFO 09-10 12:40:18 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.89 s


wavefront: .ascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascusascus updates: 10
jacobi:    . The river bank is a straight line. The river bank is 100 meters long. The fox starts at updates: 10


## 要点回顾

- overlay 任务读的是**算子三元组**（输入/输出/原权重），输出直接进模型前向。
- buffer 状态零初始化是签名约束：初始行为严格等于无补丁。
- `keys` 决定谁和谁共享；route 键由你计算，框架只看字符串。
- 混合、多头、判别器……都是"状态里多几个字段 + forward 里多几行数学"。
- wavefront/Jacobi 的差别只在 update 循环里"什么时候 add_"，发布纪律不变。

下一篇：[教程 4：三种循环](04_loops.ipynb)。